# Collaborative Filtering -- Users Like You

In notebook 1 we built a bias model: `prediction = global_mean + user_bias + item_bias`.
It improved on the global mean by capturing how generous each user is and how popular
each item is. But it cannot learn that two users share a specific taste.

**Collaborative filtering** fixes this. The core idea: users who agreed in the past will
agree in the future. If user A and user B both rated the same films highly, then a film
user A loved is probably worth recommending to user B -- even if that film has an
unremarkable average across all users.

The word "collaborative" refers to the fact that users implicitly collaborate: each
user's ratings help improve recommendations for every other user with similar taste.
No one coordinates this; it emerges from the rating data alone.

| Step | What we build | Why |
|------|--------------|-----|
| 1 | The similarity idea | The intuition behind collaborative filtering |
| 2 | User-item matrix as dicts | The data structure for efficient lookup |
| 3 | Cosine similarity | How to measure how alike two users are |
| 4 | User-user prediction | Predict ratings from neighbours' opinions |
| 5 | Item-item CF | Flip the axis: similar items, not similar users |
| 6 | The sparsity wall | Where CF breaks down |
| 7 | Scaling | Why pairwise similarity does not scale |

In [ ]:
import random, math, matplotlib.pyplot as plt
from collections import defaultdict
%matplotlib inline

def make_ratings(n_users=80, n_items=120, n_factors=4, density=0.07, seed=42):
    random.seed(seed)
    U = [[random.gauss(0,1) for _ in range(n_factors)] for _ in range(n_users)]
    V = [[random.gauss(0,1) for _ in range(n_factors)] for _ in range(n_items)]
    bu = [random.gauss(0, 0.3) for _ in range(n_users)]
    bv = [random.gauss(0, 0.3) for _ in range(n_items)]
    mu = 3.5
    ratings = []
    for u in range(n_users):
        for v in range(n_items):
            if random.random() < density:
                r = mu + bu[u] + bv[v] + sum(U[u][k]*V[v][k] for k in range(n_factors))
                r = max(1.0, min(5.0, r + random.gauss(0, 0.3)))
                ratings.append((u, v, round(r)))
    return ratings, n_users, n_items

def train_val_split(ratings, val_frac=0.2, seed=42):
    random.seed(seed)
    shuffled = list(ratings)
    random.shuffle(shuffled)
    split = int(len(shuffled) * (1 - val_frac))
    return shuffled[:split], shuffled[split:]

def rmse(predictions):  # [(pred, actual), ...]
    return math.sqrt(sum((p - a)**2 for p, a in predictions) / len(predictions))

ratings, N_USERS, N_ITEMS = make_ratings()
train, val = train_val_split(ratings)
global_mean = sum(r for u, v, r in train) / len(train)
print(f'{len(train)} train ratings,  {len(val)} val ratings,  global mean: {global_mean:.3f}')

In [ ]:
# Bias model from notebook 1 (regularised, lambda=10) -- our baseline to beat
user_rs = defaultdict(list)
item_rs = defaultdict(list)
for u, v, r in train:
    user_rs[u].append(r)
    item_rs[v].append(r)

lam = 10
user_bias = {u: sum(r - global_mean for r in rs) / (len(rs) + lam) for u, rs in user_rs.items()}
item_bias = {v: sum(r - global_mean for r in rs) / (len(rs) + lam) for v, rs in item_rs.items()}

bias_rmse = rmse([
    (global_mean + user_bias.get(u, 0) + item_bias.get(v, 0), r)
    for u, v, r in val
])
print(f'Bias model RMSE (baseline from notebook 1): {bias_rmse:.4f}')

## Step 1: The Similarity Idea

Collaborative filtering rests on one assumption: **past agreement predicts future
agreement**.

If user A and user B both rated item X highly and item Y poorly, they probably share
some underlying taste. When user A rates a new item Z highly, it is a good bet that
user B would enjoy Z too -- even if user B has never seen Z.

Nothing in this reasoning requires knowing what X, Y, or Z actually are. No genres,
no descriptions, no metadata. The pattern of who liked what is the only input. This
is what makes CF powerful: it works from pure behaviour, not content.

The approach comes in two flavours. **User-user CF** finds users similar to the target
user and borrows their opinions. **Item-item CF** finds items similar to ones the target
user liked and recommends those. Both rest on the same mathematical primitive: a
similarity score between two rating vectors.

## Step 2: Building the User-Item Matrix

We represent the training ratings in two ways:

- `user_ratings`: a `{user: {item: rating}}` dict -- O(1) to retrieve all items rated
  by a given user.
- `item_ratings`: a `{item: {user: rating}}` dict -- O(1) to retrieve all users who
  rated a given item.

Both are built from the same training data. The duplication is intentional: user-user
CF needs to look up a user's history; item-item CF needs to look up an item's raters.

In [ ]:
user_ratings = defaultdict(dict)  # {user: {item: rating}}
item_ratings = defaultdict(dict)  # {item: {user: rating}}
for u, v, r in train:
    user_ratings[u][v] = r
    item_ratings[v][u] = r

print(f'User 0 has rated {len(user_ratings[0])} items.')
print(f'Example ratings by user 0: {dict(list(user_ratings[0].items())[:5])}')

## Step 3: Cosine Similarity Between Users

How do we measure how alike two users are? Each user is a sparse vector of ratings --
a number for each item they have rated, nothing for the rest.

**Cosine similarity** measures the angle between two vectors. Vectors pointing in the
same direction (same relative ratings) give cosine 1. Opposite directions give -1.
Perpendicular vectors (no relationship) give 0.

The formula, for two users with rating vectors u and v over their **shared items** S:

```
sim(u, v) = sum_i(r_ui * r_vi) / ( sqrt(sum_i r_ui^2) * sqrt(sum_i r_vi^2) )
```

The sum runs only over items **both** users have rated. If they share no items,
similarity is 0.

The same formula works for item-item similarity: treat each item as a vector over
users (the transpose of the user-item matrix). One function handles both cases.

In [ ]:
def cosine(vec1, vec2):
    # vec1, vec2 are dicts {index: value}; similarity computed over shared indices
    shared = set(vec1) & set(vec2)
    if not shared:
        return 0.0
    dot = sum(vec1[k] * vec2[k] for k in shared)
    n1  = math.sqrt(sum(vec1[k]**2 for k in shared))
    n2  = math.sqrt(sum(vec2[k]**2 for k in shared))
    return dot / (n1 * n2) if n1 and n2 else 0.0

print('User-user similarities (cosine over shared rated items):')
for u1, u2 in [(0, 1), (0, 5), (0, 10), (0, 20)]:
    shared = set(user_ratings[u1]) & set(user_ratings[u2])
    sim = cosine(user_ratings[u1], user_ratings[u2])
    print(f'  users ({u1:2d}, {u2:2d}): sim={sim:+.3f}  shared={len(shared)} items')

## Step 4: User-User Prediction

To predict how user u would rate item v:

1. Find all training users who have rated item v.
2. Rank them by similarity to user u (highest first).
3. Take the top K neighbours.
4. Return their **similarity-weighted average rating** for item v.

Weighting by similarity means a user with similarity 0.9 counts for nine times as
much as one with similarity 0.1. If no neighbour has rated item v, fall back to the
global mean.

In [ ]:
def user_user_predict(u, v, K=5):
    neighbours = sorted(
        [(cosine(user_ratings[u], user_ratings[other]), user_ratings[other][v])
         for other in user_ratings if other != u and v in user_ratings[other]],
        reverse=True
    )[:K]
    if not neighbours:
        return global_mean
    w = sum(abs(s) for s, _ in neighbours)
    return sum(s * r for s, r in neighbours) / w if w else global_mean

uu_rmse = rmse([(user_user_predict(u, v), r) for u, v, r in val])
print(f'RMSE (user-user CF, K=5): {uu_rmse:.4f}')
print(f'RMSE (bias model):        {bias_rmse:.4f}')

## Step 5: Item-Item Collaborative Filtering

The same idea, flipped. Instead of finding users similar to the target user, we find
items similar to items the target user has already rated.

**Item-item similarity** is the cosine similarity between two items' rating vectors,
treating each item as a vector over users who rated it.

To predict how user u would rate item v:

1. Consider all items user u has already rated.
2. Compute each one's similarity to item v.
3. Take the top K most-similar items the user has rated.
4. Return the similarity-weighted average of the user's ratings for those items.

Item-item CF has a practical advantage: items are more stable than users. A user's
tastes can shift over time; an item's identity does not. Item similarities can be
precomputed offline and cached, making inference faster than user-user CF.

In [ ]:
def item_item_predict(u, v, K=5):
    if v not in item_ratings:
        return global_mean
    neighbours = sorted(
        [(cosine(item_ratings[v], item_ratings[i]), user_ratings[u][i])
         for i in user_ratings[u] if i != v and i in item_ratings],
        reverse=True
    )[:K]
    if not neighbours:
        return global_mean
    w = sum(abs(s) for s, _ in neighbours)
    return sum(s * r for s, r in neighbours) / w if w else global_mean

ii_rmse = rmse([(item_item_predict(u, v), r) for u, v, r in val])
print(f'RMSE (item-item CF, K=5): {ii_rmse:.4f}')
print(f'RMSE (user-user CF, K=5): {uu_rmse:.4f}')
print(f'RMSE (bias model):        {bias_rmse:.4f}')

## Step 6: The Sparsity Wall

CF relies on finding users (or items) with meaningful overlap. With 7% density and
120 items, two users share on average `0.07 * 0.07 * 120 ≈ 0.6` items. Many user
pairs share zero items.

Now consider a new user with only 2 ratings. What happens?

In [ ]:
# Simulate a new user who has rated only 2 items
new_user = {5: 4, 12: 5}

# User-user CF: which training users share at least one item with this new user?
overlapping = [
    (cosine(new_user, user_ratings[u]), u)
    for u in user_ratings
    if set(new_user) & set(user_ratings[u])
]
overlapping.sort(reverse=True)
print(f'Training users with any shared item: {len(overlapping)} / {N_USERS}')
if overlapping:
    best_sim, best_u = overlapping[0]
    n_shared = len(set(new_user) & set(user_ratings[best_u]))
    print(f'Best similarity: {best_sim:.3f}  (user {best_u}, {n_shared} shared item(s))')
else:
    print('No training user shares even one item -> user-user CF gives global mean for all items.')

# Item-item CF: can we find meaningful neighbours for a typical target item?
target = 50
sim5  = cosine(item_ratings.get(target, {}), item_ratings.get(5, {}))
sim12 = cosine(item_ratings.get(target, {}), item_ratings.get(12, {}))
print(f'\nItem-item CF: similarities between target item {target} and the 2 rated items:')
print(f'  sim(item {target}, item  5) = {sim5:.3f}')
print(f'  sim(item {target}, item 12) = {sim12:.3f}')
if sim5 == 0 and sim12 == 0:
    print(f'  Zero shared raters -> similarity is 0 -> falls back to global mean ({global_mean:.2f})')

## Step 7: Scaling

Even when data is dense enough, there is a computational problem.

To find the K most similar users for a single target user, we compare that user against
every other user. With U users each having rated T items on average, one similarity
computation costs O(T). Finding the top K for one user costs O(U * T). Doing it for
all U users costs **O(U^2 * T)**.

For a real system with 1 million users and 10,000 items:

```
1,000,000^2 * 10,000 = 10^16 operations
```

That is 10 quadrillion comparisons. User-user CF cannot be a production system at scale.

Item-item CF is more tractable because items are fewer and similarities can be precomputed
offline. But it still scales as O(I^2 * U) and hits the same wall at the scale of a
major streaming service.

The key insight needed to break through: instead of comparing users in rating space,
represent them as short, dense vectors in a **latent factor space** and learn those
vectors from data. Two users with similar taste end up close in that space even if they
share few rated items. That is matrix factorization -- the subject of the next notebook.

## Summary

| Method | What it computes | Strength | Weakness |
|--------|-----------------|----------|----------|
| User-user CF | Cosine similarity between user rating vectors | Personalised; exploits taste communities | Fails for sparse users; O(U^2 * T) |
| Item-item CF | Cosine similarity between item rating vectors | More stable than user similarities; cacheable | Fails for sparse items; O(I^2 * U) |

Both methods share a fatal flaw: they need **overlap** to compute similarity. A user
with 2 ratings has almost no overlap with anyone. The sparsity that defines recommendation
systems is precisely what makes similarity-based approaches unreliable.

## Your turn

**1. Vary K.** Change K in `user_user_predict` from 1 to 5 to 20. How does val RMSE
change? Why might too many neighbours hurt? Think about what happens when you include
users with low similarity in the weighted average.

**2. Minimum overlap threshold.** Modify `cosine` to return 0 if the two vectors share
fewer than 3 keys. Does enforcing this quality gate improve or worsen val RMSE?
Think about the trade-off: fewer but more reliable neighbours vs more but noisier ones.

**3. Hybrid model.** Combine the bias baseline from notebook 1 with item-item CF:
`prediction = bias_prediction + alpha * (item_cf_prediction - global_mean)`. Try
`alpha = 0.3, 0.5, 0.7`. Does the hybrid beat either model alone on val RMSE?

> Next: [Matrix Factorization](3_matrix_factorization.ipynb)